# Lesson 28: TensorFlow/Keras neural network demonstration - part 1

## Introduction to TensorFlow and Keras

**TensorFlow** is an open-source numerical computation library developed by Google. It represents computations as a directed graph of operations on tensors (multi-dimensional arrays), and handles the low-level work of executing those operations efficiently on available hardware. It also provides automatic differentiation, which makes computing gradients for backpropagation automatic rather than something you have to derive by hand.

**Keras** is a high-level neural network API that ships as part of TensorFlow (`tensorflow.keras`). It sits on top of TensorFlow and provides an 'easy' interface for defining model architectures, training loops, and evaluation. In practice, you write Keras code and TensorFlow does the underlying computation.

### How the layers fit together

```text
        ┌─────────────────────────────────────┐
        │          Your Python code           │
        │    (define model, call .fit())      │
        └──────────────────┬──────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────┐
        │                Keras                │
        │   (layers, losses, optimizers,      │
        │           training loop)            │
        └──────────────────┬──────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────┐
        │        TensorFlow Python API        │
        │    (computation graph, autodiff,    │
        │             op dispatch)            │
        └──────────────────┬──────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────┐
        │       TensorFlow C++ runtime        │
        │   (compiled ops: matmul, conv,      │
        │        activations, autodiff)       │
        └──────────────────┬──────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────┐
        │           Hardware backend          │
        └──────────────────┬──────────────────┘
                  ┌────────┴────────┐
                  │                 │
                  ▼                 ▼
           ┌─────────────┐   ┌─────────────┐
           │   CUDA GPU  │   │     CPU     │
           │  (parallel  │   │ (fallback / │
           │ matrix ops) │   │small models)│
           └─────────────┘   └─────────────┘
```

When you call `model.fit()`, Keras translates your high-level instructions into TensorFlow Python API calls. Those calls are dispatched to TensorFlow's C++ runtime, where the actual numerical operations (matrix multiplications, convolutions, activation functions, and so on) are executed as compiled C++ code. The C++ runtime then hands off the work to whichever hardware backend is available: CUDA for a compatible NVIDIA GPU, or the CPU otherwise. The gradient computation for backpropagation is handled automatically by TensorFlow's autodiff engine at the C++ level, so neither you nor Keras needs to implement it explicitly.

## Notebook set up

### Imports

In [ ]:
# Third party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers, Model

tf.random.set_seed(315)

## 1. Data preparation

### 1.1. Load California housing data

In [ ]:
housing_df = pd.read_csv('https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/refs/heads/main/data/california_housing.csv')
housing_df.head()

In [ ]:
housing_df.info()

In [ ]:
label = 'MedHouseVal'
features = ['MedInc','HouseAge','AveRooms','AveBedrms','Population','AveOccup','Latitude','Longitude']

### 1.2. Train test split

In [ ]:
training_df, testing_df = train_test_split(housing_df, random_state=42)

### 1.3. Standard scale

#### Features

In [ ]:
feature_scaler = StandardScaler()
feature_scaler.fit(training_df[features])

training_df[features] = feature_scaler.transform(training_df[features])
testing_df[features] = feature_scaler.transform(testing_df[features])

#### Label

In [ ]:
label_scaler = StandardScaler()
label_scaler.fit(training_df[label].to_frame())

training_df[label] = label_scaler.transform(training_df[label].to_frame())
testing_df[label] = label_scaler.transform(testing_df[label].to_frame())

### 1.4. Handle outliers

In [ ]:
for feature in features:

    q1 = training_df[feature].quantile(0.25)
    q3 = training_df[feature].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    training_df[feature] = training_df[feature].clip(lower=lower_bound, upper=upper_bound)
    testing_df[feature] = testing_df[feature].clip(lower=lower_bound, upper=upper_bound)

## 2. Linear regression baseline

### 2.1. Fit

In [ ]:
linear_model = LinearRegression(n_jobs=-1)
fit_result = linear_model.fit(training_df[features], training_df[label])

### 2.2. Test set evaluation

In [ ]:
linear_predictions = linear_model.predict(testing_df[features])
linear_rsquared = linear_model.score(testing_df[features], testing_df[label])
print(f'Linear regression R² on test set: {linear_rsquared:.4f}')

## 3. Keras Sequential API model

The Sequential API is the simplest way to build a neural network in Keras. It allows you to create models layer-by-layer in a linear stack.

### 3.1. Build model

In [ ]:
sequential_model = keras.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1)
])

sequential_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='mse',
    metrics=['mae']
)

sequential_model.summary()

### 3.2. Train model

In [ ]:
sequential_history = sequential_model.fit(
    training_df[features],
    training_df[label],
    epochs=75,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

print('Training complete.')
print(f'Final training loss: {sequential_history.history["loss"][-1]:.4f}')
print(f'Final validation loss: {sequential_history.history["val_loss"][-1]:.4f}')

### 3.3. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('Sequential API: loss')
axes[0].plot(sequential_history.history['loss'], label='Training')
axes[0].plot(sequential_history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend(loc='best')

axes[1].set_title('Sequential API: MAE')
axes[1].plot(sequential_history.history['mae'], label='Training')
axes[1].plot(sequential_history.history['val_mae'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean absolute error')
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()

### 3.4. Test set evaluation

In [ ]:
sequential_predictions = sequential_model.predict(testing_df[features], verbose=0).flatten()

ss_res = np.sum((testing_df[label] - sequential_predictions) ** 2)
ss_tot = np.sum((testing_df[label] - np.mean(testing_df[label])) ** 2)
sequential_rsquared = 1 - (ss_res / ss_tot)

print(f'Sequential API model R² on test set: {sequential_rsquared:.4f}')

### 3.5. Performance analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].set_title('Sequential API predictions')
axes[0].scatter(
    testing_df[label], sequential_predictions,
    c='black', s=0.5, alpha=0.5
)
axes[0].plot(
    [testing_df[label].min(), testing_df[label].max()],
    [testing_df[label].min(), testing_df[label].max()],
    color='red', linestyle='--'
)
axes[0].set_xlabel('True values (standardized)')
axes[0].set_ylabel('Predicted values (standardized)')

axes[1].set_title('Residuals vs predicted values')
axes[1].scatter(
    sequential_predictions, testing_df[label] - sequential_predictions,
    c='black', s=0.5, alpha=0.5
)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted values (standardized)')
axes[1].set_ylabel('Residuals (standardized)')

plt.tight_layout()
plt.show()

## 4. Keras Functional API model

The Functional API provides more flexibility than the Sequential API. It allows you to create models with non-linear topology, shared layers, and multiple inputs or outputs.

### 4.1. Build model

In [ ]:
# Define input layer
inputs = keras.Input(shape=(8,), name='input_features')

# Define hidden layers
x = layers.Dense(64, activation='relu', name='hidden_1')(inputs)
x = layers.Dropout(0.2, name='dropout_1')(x)
x = layers.Dense(32, activation='relu', name='hidden_2')(x)
x = layers.Dropout(0.2, name='dropout_2')(x)

# Define output layer
outputs = layers.Dense(1, name='output')(x)

# Create model
functional_model = Model(inputs=inputs, outputs=outputs, name='functional_mlp')

functional_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='mse',
    metrics=['mae']
)

functional_model.summary()

### 4.2. Train model

In [ ]:
functional_history = functional_model.fit(
    training_df[features],
    training_df[label],
    epochs=75,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

print('Training complete.')
print(f'Final training loss: {functional_history.history["loss"][-1]:.4f}')
print(f'Final validation loss: {functional_history.history["val_loss"][-1]:.4f}')

### 4.3. Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('Functional API: loss')
axes[0].plot(functional_history.history['loss'], label='Training')
axes[0].plot(functional_history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend(loc='best')

axes[1].set_title('Functional API: MAE')
axes[1].plot(functional_history.history['mae'], label='Training')
axes[1].plot(functional_history.history['val_mae'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean absolute error')
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()

### 4.4. Test set evaluation

In [ ]:
functional_predictions = functional_model.predict(testing_df[features], verbose=0).flatten()

ss_res = np.sum((testing_df[label] - functional_predictions) ** 2)
ss_tot = np.sum((testing_df[label] - np.mean(testing_df[label])) ** 2)
functional_rsquared = 1 - (ss_res / ss_tot)

print(f'Functional API model R² on test set: {functional_rsquared:.4f}')

### 4.5. Performance analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].set_title('Functional API predictions')
axes[0].scatter(
    testing_df[label], functional_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[0].plot(
    [testing_df[label].min(), testing_df[label].max()],
    [testing_df[label].min(), testing_df[label].max()],
    color='red', linestyle='--'
)

axes[0].set_xlabel('True values (standardized)')
axes[0].set_ylabel('Predicted values (standardized)')

axes[1].set_title('Residuals vs predicted values')
axes[1].scatter(
    functional_predictions, testing_df[label] - functional_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted values (standardized)')
axes[1].set_ylabel('Residuals (standardized)')

plt.tight_layout()
plt.show()

## 5. Model comparison

In [ ]:
print(f'Linear regression R² on test set: {linear_rsquared:.4f}')
print(f'Sequential API model R² on test set: {sequential_rsquared:.4f}')
print(f'Functional API model R² on test set: {functional_rsquared:.4f}')

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 9))

# Linear regression
axes[0, 0].set_title('Linear regression predictions')
axes[0, 0].scatter(
    testing_df[label], linear_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[0, 0].plot(
    [testing_df[label].min(), testing_df[label].max()],
    [testing_df[label].min(), testing_df[label].max()],
    color='red', linestyle='--'
)

axes[0, 0].set_xlabel('True values (standardized)')
axes[0, 0].set_ylabel('Predicted values (standardized)')

axes[0, 1].set_title('Linear regression residuals')
axes[0, 1].scatter(
    linear_predictions, testing_df[label] - linear_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].set_xlabel('Predicted values (standardized)')
axes[0, 1].set_ylabel('Residuals (standardized)')

# Sequential API
axes[1, 0].set_title('Sequential API predictions')
axes[1, 0].scatter(
    testing_df[label], sequential_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[1, 0].plot(
    [testing_df[label].min(), testing_df[label].max()],
    [testing_df[label].min(), testing_df[label].max()],
    color='red', linestyle='--'
)

axes[1, 0].set_xlabel('True values (standardized)')
axes[1, 0].set_ylabel('Predicted values (standardized)')

axes[1, 1].set_title('Sequential API residuals')
axes[1, 1].scatter(
    sequential_predictions, testing_df[label] - sequential_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[1, 1].axhline(0, color='red', linestyle='--')
axes[1, 1].set_xlabel('Predicted values (standardized)')
axes[1, 1].set_ylabel('Residuals (standardized)')

# Functional API
axes[2, 0].set_title('Functional API predictions')
axes[2, 0].scatter(
    testing_df[label], functional_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[2, 0].plot(
    [testing_df[label].min(), testing_df[label].max()],
    [testing_df[label].min(), testing_df[label].max()],
    color='red', linestyle='--'
)

axes[2, 0].set_xlabel('True values (standardized)')
axes[2, 0].set_ylabel('Predicted values (standardized)')

axes[2, 1].set_title('Functional API residuals')
axes[2, 1].scatter(
    functional_predictions, testing_df[label] - functional_predictions,
    c='black', s=0.5, alpha=0.5
)

axes[2, 1].axhline(0, color='red', linestyle='--')
axes[2, 1].set_xlabel('Predicted values (standardized)')
axes[2, 1].set_ylabel('Residuals (standardized)')

plt.tight_layout()
plt.show()